In [1]:
# --- Cell 1: imports ---
%load_ext autoreload
%autoreload 2
import sys, torch, numpy as np
sys.path.append('..')
sys.path.append('../..')

from common.seed import set_seed
from common.io_utils import save_results, load_results
from shared.pacs import CLASSES
from shared.pacs_protocol import build_loaders
from task2.backbones.backbone import PACSModel
from task2.evaluations.metrics import evaluate_loader, evaluate_source_val
from task2.evaluations.class_analysis import (per_class_table, class_deltas,
                                              dominant_confusions, predicted_distribution)
from task3.evaluations.source_domain_separability import (collect_source_features,
                                                          source_domain_separability)
from task3.evaluations.sharpness import fixed_val_batch, sharpness_proxy

set_seed(6304)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = '../../datasets/PACS'

METHODS = ['erm', 'dan_dg', 'sam']
CKPT = {'erm':    '../../datasets/cache/task2_source_only.pt',   # reused UNCHANGED
        'dan_dg': '../../datasets/cache/task3_dan_dg.pt',
        'sam':    '../../datasets/cache/task3_sam.pt'}

In [2]:
# --- Cell 2: loaders — target included ONLY here, in the final evaluation ---
splits = load_results('../../shared/splits/pacs_sketch_seed6304.json')
loaders = build_loaders(DATA_ROOT, splits, batch_per_source=8, target_batch=24,
                        eval_batch=64, num_workers=0, seed=6304, include_target=True)
print('sketch images:', len(loaders['target_eval'].dataset))

sketch images: 3929


In [3]:
# --- Cell 3: source-side metrics, Sketch, separability, sharpness ---
xb, yb = fixed_val_batch(loaders['source_val'], device, per_domain=32, seed=6304)
print('fixed sharpness batch:', tuple(xb.shape), '(expect 96 x 3 x 224 x 224)\n')

results = {}
for m in METHODS:
    ck = torch.load(CKPT[m], map_location=device, weights_only=False)
    model = PACSModel(num_classes=7).to(device)
    model.load_state_dict(ck['model'])

    sv = evaluate_source_val(model, loaders['source_val'], device)
    f1s = [v['macro_f1'] for v in sv['per_domain'].values()]
    tg = evaluate_loader(model, loaders['target_eval'], device)

    feats, doms, n_per = collect_source_features(model, loaders['source_val'], device)
    sep = source_domain_separability(feats, doms, seed=6304)
    sharp = sharpness_proxy(model, xb, yb, rho=0.05)

    results[m] = {'best_epoch': ck['best_epoch'], 'source_val': sv,
                  'worst_domain_f1': float(min(f1s)),
                  'sketch': {'accuracy': tg['accuracy'], 'macro_f1': tg['macro_f1']},
                  'separability': sep, 'sharpness': sharp,
                  '_preds': tg['preds'], '_labels': tg['labels']}
    print(f"{m:8s} src_mean_f1={sv['mean_macro_f1']:.4f} worst={min(f1s):.4f} "
          f"sketch_acc={tg['accuracy']:.4f} sketch_f1={tg['macro_f1']:.4f} "
          f"sep={sep['separability']:.4f} (chance .333) "
          f"sharp={sharp['delta_sharp']:+.4f}")

fixed sharpness batch: (96, 3, 224, 224) (expect 96 x 3 x 224 x 224)

erm      src_mean_f1=0.9385 worst=0.8947 sketch_acc=0.6373 sketch_f1=0.6557 sep=0.9269 (chance .333) sharp=+0.2897
dan_dg   src_mean_f1=0.0507 worst=0.0421 sketch_acc=0.0407 sketch_f1=0.0112 sep=0.3322 (chance .333) sharp=+0.0352
sam      src_mean_f1=0.9484 worst=0.9307 sketch_acc=0.6615 sketch_f1=0.6972 sep=0.9003 (chance .333) sharp=+0.0947


In [4]:
# --- Cell 4: the required comparison table ---
import pandas as pd
base = results['erm']['sketch']['accuracy']
rows = []
for m in METHODS:
    r = results[m]
    pd_ = r['source_val']['per_domain']
    rows.append({'method': m, 'best_epoch': r['best_epoch'],
                 **{f'{d}_f1': round(pd_[d]['macro_f1'], 4) for d in pd_},
                 'mean_f1': round(r['source_val']['mean_macro_f1'], 4),
                 'worst_f1': round(r['worst_domain_f1'], 4),
                 'sketch_acc': round(r['sketch']['accuracy'], 4),
                 'sketch_f1': round(r['sketch']['macro_f1'], 4),
                 'sketch_delta': round(r['sketch']['accuracy'] - base, 4),
                 'src_separability': round(r['separability']['separability'], 4),
                 'sharpness': round(r['sharpness']['delta_sharp'], 4)})
print(pd.DataFrame(rows).to_string(index=False))

method  best_epoch  photo_f1  art_painting_f1  cartoon_f1  mean_f1  worst_f1  sketch_acc  sketch_f1  sketch_delta  src_separability  sharpness
   erm           8    0.9715           0.8947      0.9492   0.9385    0.8947      0.6373     0.6557        0.0000            0.9269     0.2897
dan_dg           1    0.0585           0.0514      0.0421   0.0507    0.0421      0.0407     0.0112       -0.5966            0.3322     0.0352
   sam           9    0.9595           0.9307      0.9551   0.9484    0.9307      0.6615     0.6972        0.0242            0.9003     0.0947


In [5]:
# --- Cell 5: per-class Sketch changes vs ERM, with Task 2 comparison ---
tables = {m: per_class_table(results[m]['_preds'], results[m]['_labels'], CLASSES)
          for m in METHODS}
print('support:', {c: tables['erm'][c]['support'] for c in CLASSES}, '\n')
for m in METHODS:
    print(f"--- {m}")
    print('  acc :', {c: round(tables[m][c]['accuracy'], 3) for c in CLASSES})
    print('  pred:', predicted_distribution(results[m]['_preds'], CLASSES))
    if m != 'erm':
        d = class_deltas(tables['erm'], tables[m], CLASSES)
        r = sorted(d.items(), key=lambda kv: kv[1]['delta'])
        print(f"  worst {r[0][0]} {r[0][1]['delta']:+.3f} | best {r[-1][0]} {r[-1][1]['delta']:+.3f}")
    print()

# the manual asks for a comparison with the corresponding Task 2 results
t2 = load_results('../results/../../task2/results/final_eval.json')
print('Task 2 sketch acc:', {r['method']: r['tgt_acc'] for r in t2['table']})

support: {'dog': 772, 'elephant': 740, 'giraffe': 753, 'guitar': 608, 'horse': 816, 'house': 80, 'person': 160} 

--- erm
  acc : {'dog': 0.455, 'elephant': 0.969, 'giraffe': 0.587, 'guitar': 0.837, 'horse': 0.451, 'house': 0.875, 'person': 0.294}
  pred: {'dog': 634, 'elephant': 1602, 'giraffe': 536, 'guitar': 518, 'horse': 509, 'house': 77, 'person': 53}

--- dan_dg
  acc : {'dog': 0.0, 'elephant': 0.0, 'giraffe': 0.0, 'guitar': 0.0, 'horse': 0.0, 'house': 0.0, 'person': 1.0}
  pred: {'dog': 0, 'elephant': 0, 'giraffe': 0, 'guitar': 0, 'horse': 0, 'house': 0, 'person': 3929}
  worst elephant -0.969 | best person +0.706

--- sam
  acc : {'dog': 0.501, 'elephant': 0.959, 'giraffe': 0.624, 'guitar': 0.743, 'horse': 0.499, 'house': 0.887, 'person': 0.637}
  pred: {'dog': 767, 'elephant': 1392, 'giraffe': 577, 'guitar': 453, 'horse': 490, 'house': 74, 'person': 176}
  worst guitar -0.094 | best person +0.344

Task 2 sketch acc: {'source_only': 0.6373, 'dan': 0.6281, 'dann': 0.0812, 'cdan'

In [6]:
# --- Cell 6: save ---
save_results({'step': 'task3_final_eval', 'seed': 6304, 'classes': list(CLASSES),
              'table': rows,
              'per_class': tables,
              'deltas': {m: class_deltas(tables['erm'], tables[m], CLASSES)
                         for m in METHODS if m != 'erm'},
              'confusions': {m: dominant_confusions(results[m]['_preds'],
                                                    results[m]['_labels'], CLASSES)
                             for m in METHODS},
              'separability': {m: results[m]['separability'] for m in METHODS},
              'sharpness': {m: results[m]['sharpness'] for m in METHODS}},
             '../results/final_eval.json')
print('saved')

saved
